# ⚖️ Monster Breach — The Judge

You are a **Pipeline Ranger**. The Corruption Horde has poisoned the crystal mines of
Datapolis. Each monster guards two data defects; you defeat it by building a **real
Fabric Data Pipeline** that cleans the crystals, then let the Judge inspect your work.

## How to play
1. **Attach** `MonsterBreach_LH` as the default Lakehouse on this notebook.
2. **Run cells 1–3** below to wake the Judge.
3. Call `judge.help()` to see the monster roster.
4. For each level:
   - `judge.briefing(N)` → reads the case file (story + task + the **output table name** to produce + hint).
   - Open the matching **empty pipeline** (e.g. `Lvl01_CopyQuest`) in the Fabric UI and build it so it
     reads the dirty table and writes the cleaned crystals to the **required Lakehouse table**.
   - **Run the pipeline** in the Fabric UI.
   - `judge.check_level(N)` → the Judge diffs your output table against the secret oracle, scores it,
     and ships a `BattleEvents` telemetry event.
5. `judge.progress()` → monsters defeated + total score + rank.

> The Judge only cares about your **result table** — any technique that produces the correct
> cleaned crystals counts as a victory. The briefing recommends the activities to *learn*.

## ⚙️ Step 0 — Player

In [ ]:
# --- EDIT THIS ---
PLAYER_NAME = "Your Name Here"   # shown on your shareable badge at the end
# -----------------
print(f"Ranger: {PLAYER_NAME}")

## Step 1 — Setup

In [ ]:
import os, uuid, json, math, datetime as dt, requests
from IPython.display import display, Markdown
from pyspark.sql import functions as F

WORKSPACE_ID = None
try:
    import notebookutils
    WORKSPACE_ID = notebookutils.runtime.context.get("currentWorkspaceId")
except Exception:
    try:
        import mssparkutils
        WORKSPACE_ID = mssparkutils.runtime.context.get("currentWorkspaceId")
    except Exception:
        pass

LH_NAME  = "MonsterBreach_LH"
EH_NAME  = "BattleData"          # KQL database display name
EH_TABLE = "BattleEvents"

SESSION_ID = str(uuid.uuid4())
PLAYER_ID  = os.environ.get("USER", "ranger")

print(f"Workspace: {WORKSPACE_ID}")
print(f"Session:   {SESSION_ID}")

## Step 2 — Helpers (Lakehouse diff + Eventhouse telemetry)

In [ ]:
def _token(resource: str) -> str:
    try:
        import notebookutils
        return notebookutils.credentials.getToken(resource)
    except Exception:
        import mssparkutils
        return mssparkutils.credentials.getToken(resource)

def _fabric_get(url: str) -> dict:
    tok = _token("pbi")
    r = requests.get(url, headers={"Authorization": f"Bearer {tok}"}, timeout=60)
    r.raise_for_status()
    return r.json()

# --- Eventhouse query URI resolution + telemetry --------------------
_EH_QSI = {"v": None}
def _eh_query_uri() -> str:
    if _EH_QSI["v"]:
        return _EH_QSI["v"]
    dbs = _fabric_get(f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/kqlDatabases").get("value", [])
    target = next((d for d in dbs if d["displayName"] == EH_NAME), None)
    if not target:
        raise RuntimeError(f"KQL DB '{EH_NAME}' not found — was arcade.install run?")
    info = _fabric_get(f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/kqlDatabases/{target['id']}")
    uri = info.get("properties", {}).get("queryServiceUri")
    if not uri:
        raise RuntimeError("No queryServiceUri for BattleData.")
    _EH_QSI["v"] = uri
    return uri

def log_event(event_type, level=0, monster="", activity="", score=0,
              crystals_saved=0, rows_processed=0, validation_result="INFO"):
    ts = dt.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.%fZ")
    eid = str(uuid.uuid4())
    # EventId,Timestamp,SessionId,PlayerId,EventType,Level,Monster,ActivityUsed,
    # Score,CrystalsSaved,RowsProcessed,ValidationResult
    row = (f'"{eid}","{ts}","{SESSION_ID}","{PLAYER_ID}","{event_type}",'
           f'{level},"{monster}","{activity}",{score},{crystals_saved},'
           f'{rows_processed},"{validation_result}"')
    csl = f".ingest inline into table {EH_TABLE} <|\n{row}"
    try:
        tok = _token("kusto")
        requests.post(f"{_eh_query_uri()}/v1/rest/mgmt",
                      headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json"},
                      json={"db": EH_NAME, "csl": csl}, timeout=15)
    except Exception as e:
        print(f"  ⚠️ telemetry failed: {e}")

def query_kql(kql: str):
    tok = _token("kusto")
    r = requests.post(f"{_eh_query_uri()}/v1/rest/query",
                      headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json"},
                      json={"db": EH_NAME, "csl": kql}, timeout=30)
    r.raise_for_status()
    tbl = r.json()["Tables"][0]
    cols = [c["ColumnName"] for c in tbl["Columns"]]
    return [dict(zip(cols, row)) for row in tbl["Rows"]]

# --- Lakehouse table diff ------------------------------------------
def _table_exists(name: str) -> bool:
    try:
        spark.table(name)
        return True
    except Exception:
        return False

def diff_tables(player_tbl: str, expected_tbl: str):
    '''Compare player output to the oracle. Returns (passed, report dict).'''
    exp = spark.table(expected_tbl)
    got = spark.table(player_tbl)
    exp_n = exp.count()
    got_n = got.count()
    # Key set comparison on CrystalId (the stable business key in every level).
    exp_ids = {r[0] for r in exp.select("CrystalId").distinct().collect()}
    got_ids = {r[0] for r in got.select("CrystalId").distinct().collect()} if "CrystalId" in got.columns else set()
    missing = exp_ids - got_ids
    extra   = got_ids - exp_ids
    rep = {"expected_rows": exp_n, "player_rows": got_n,
           "missing": len(missing), "extra": len(extra)}
    passed = (got_n == exp_n) and (not missing) and (not extra)
    return passed, rep

print("✅ Helpers loaded — log_event, query_kql, diff_tables.")

## Step 3 — Monster roster + Judge

In [ ]:
# Monster roster. The Judge is technique-agnostic: it only checks that your
# pipeline produced the correct OUTPUT table. The recommended activities are what
# the level is designed to teach. Each level combines TWO data-engineering skills.
LEVELS = {
    1: {
        "name": "Copy Quest", "monster": "\U0001F41B Null Bug", "concept": "Copy Activity + Schema cast",
        "points": 100, "pipeline": "Lvl01_CopyQuest",
        "dirty": "dirty_crystals_lvl01", "expected": "clean_expected_lvl01",
        "output": "player_lvl01",
        "narrative": ("The Null Bug devoured many crystal IDs while the Schema Dragon scorched the "
                      "type metadata: `WeightText` is plain text and `PurityText` looks like "
                      "\"85%\". Rows with a NULL `CrystalId` are worthless slag."),
        "task": ("Produce `player_lvl01` with columns `CrystalId`, `Mine`, `Weight` (double) and "
                 "`Purity` (double 0\u20131): drop every row where `CrystalId` is NULL, cast "
                 "`WeightText` to double, and parse `PurityText` (strip `%`, divide by 100)."),
        "hint": ("Use a **Copy Activity** (or **Dataflow Gen2**) reading `dirty_crystals_lvl01`. "
                 "Keep `CrystalId IS NOT NULL`, change `WeightText` type to decimal, and add a "
                 "column `Number.FromText(Text.Remove([PurityText],\"%\"))/100`. Output `player_lvl01`."),
    },
    2: {
        "name": "Filter Fortress", "monster": "\U0001F47B Duplicate Ghost", "concept": "Distinct + Latest batch",
        "points": 100, "pipeline": "Lvl02_FilterFortress",
        "dirty": "dirty_crystals_lvl02", "expected": "clean_expected_lvl02",
        "output": "player_lvl02",
        "narrative": ("Two minions strike at once, in two SEPARATE ways. The **Duplicate Ghost** echoes "
                      "each crystal 1\u20133 times as **identical copies** \u2014 same `CrystalId`, same "
                      "`IngestDate`, same everything. Meanwhile the **Latency Demon** mixes four "
                      "`IngestDate` batches together: every crystal belongs to exactly ONE batch, and "
                      "only the **most recent** batch (the max `IngestDate`) is fresh. Note: a crystal "
                      "never appears under two different dates \u2014 the two defects are independent."),
        "task": ("Defeat BOTH defects to produce `player_lvl02`: (1) collapse the identical duplicate "
                 "rows left by the Ghost, and (2) drop the stale batches, keeping only the rows whose "
                 "`IngestDate` equals the **latest** batch. The result is the **distinct** rows of "
                 "`dirty_crystals_lvl02` that belong to the most recent `IngestDate`."),
        "hint": ("Build a **Dataflow Gen2**: aggregate `MAX(IngestDate)` and filter `IngestDate = max` "
                 "to banish the Latency Demon's stale batches, then a **Remove duplicates / Distinct** "
                 "on all columns to kill the Ghost's copies. Both steps are needed and the order does "
                 "not matter. Output to `player_lvl02`."),
    },
    3: {
        "name": "Branch Lair", "monster": "\U0001F3AD Branch Mimic", "concept": "If Condition + Switch",
        "points": 100, "pipeline": "Lvl03_BranchLair",
        "dirty": "dirty_crystals_lvl03", "expected": "clean_expected_lvl03",
        "output": "player_lvl03",
        "narrative": ("The Branch Mimic disguises corrupted crystals as valid ones and grades them "
                      "A/B/C/F. Only crystals whose `Status` is `Valid` **and** whose `Grade` is not "
                      "`F` may reach the vault."),
        "task": "Produce `player_lvl03` containing only rows where `Status = 'Valid'` AND `Grade <> 'F'`.",
        "hint": ("Use an **If Condition** on `Status` and a **Switch** on `Grade` (A/B/C \u2192 keep, "
                 "F \u2192 discard). A single filter `Status = 'Valid' AND Grade <> 'F'` produces the "
                 "same result. Output to `player_lvl03`."),
    },
    4: {
        "name": "ForEach Reef", "monster": "\U0001F300 Loop Wraith", "concept": "ForEach + parameters + Retry",
        "points": 100, "pipeline": "Lvl04_ForEachReef",
        "dirty": "dirty_crystals_lvl04", "expected": "clean_expected_lvl04",
        "output": "player_lvl04",
        "narrative": ("The Loop Wraith scattered crystals across five mines and the Failure Phantom "
                      "corrupted the scales with negative and **under-weight** readings. Each mine has "
                      "its OWN minimum safe weight, published in the `mine_thresholds` table \u2014 a "
                      "crystal is valid only if its `Weight` exceeds the threshold **of its own mine**. "
                      "A blanket `Weight > 0` lets too many junk crystals through and the vault rejects it."),
        "task": ("Produce `player_lvl04` keeping, for each `Mine`, only the rows whose `Weight` is "
                 "greater than that mine's `MinWeight` in `mine_thresholds`. Output columns: "
                 "`CrystalId`, `Mine`, `Weight`, `Purity`."),
        "hint": ("Read the per-mine limits from `mine_thresholds`, then **ForEach** mine: pass the mine "
                 "and its `MinWeight` as **parameters** to an inner Copy/Dataflow filtering "
                 "`Mine = @item().Mine AND Weight > @item().MinWeight`, with a **Retry policy** on the "
                 "activity. (A single **join** between `dirty_crystals_lvl04` and `mine_thresholds` "
                 "keeping `Weight > MinWeight` also works \u2014 the Judge only checks the result.) "
                 "Output to `player_lvl04`."),
    },
    5: {
        "name": "Boss Battle", "monster": "\U0001F451 Corruption King", "concept": "Mega-pipeline: every defect",
        "points": 200, "pipeline": "BossBattle_CorruptionKing",
        "dirty": "dirty_crystals_boss", "expected": "clean_expected_boss",
        "output": "player_boss",
        "narrative": ("The Corruption King unleashes every minion at once: NULL ids, duplicates, "
                      "stale batches, corrupted status, F-grade rejects and **under-weight** crystals "
                      "\u2014 and each mine still enforces its own `MinWeight` from `mine_thresholds` "
                      "\u2014 all infest `dirty_crystals_boss`. Forge one mega-pipeline to cleanse them all."),
        "task": ("Produce `player_boss` containing rows from the **latest** `IngestDate` that satisfy "
                 "ALL of: `CrystalId` not null, `Status = 'Valid'`, `Grade <> 'F'`, and `Weight` greater "
                 "than that mine's `MinWeight` in `mine_thresholds`, then **distinct** rows."),
        "hint": ("Chain every technique into one pipeline: Lookup `MAX(IngestDate)` \u2192 filter to it "
                 "\u2192 drop NULL ids, keep `Status='Valid'` and `Grade<>'F'` \u2192 **join "
                 "`mine_thresholds`** (or ForEach mine) keeping `Weight > MinWeight` \u2192 **Distinct**. "
                 "Output to `player_boss`."),
    },
}

RANKS = [
    (0,   "Crystal Recruit"),
    (100, "Pipe Apprentice"),
    (300, "Data Warden"),
    (500, "Corruption Hunter"),
    (600, "\U0001F3C6 Horde Vanquisher"),
]

BOSS_LEVEL = 5

def rank_for(total):
    r = RANKS[0][1]
    for thr, name in RANKS:
        if total >= thr:
            r = name
    return r

print(f"\U0001F409 {len(LEVELS)} monsters registered. Max score: "
      f"{sum(v['points'] for v in LEVELS.values())}.")

In [ ]:
class Judge:
    def help(self):
        rows = ["### \u2696\uFE0F Monster roster\n",
                "| Lvl | Monster | Skills to learn | Pipeline | Output table |",
                "|---|---|---|---|---|"]
        for n, d in LEVELS.items():
            rows.append(f"| {n} | {d['monster']} | {d['concept']} | `{d['pipeline']}` | `{d['output']}` |")
        rows.append("\n**Commands:** `judge.briefing(N)`, `judge.check_level(N)`, `judge.progress()`")
        display(Markdown("\n".join(rows)))

    def briefing(self, level):
        d = LEVELS.get(level)
        if not d:
            print(f"\u2753 unknown level {level}"); return
        md = [f"## Level {level} \u2014 {d['name']}  ({d['monster']})",
              f"**Skills:** {d['concept']}  \u00B7  **Reward:** {d['points']} pts",
              "",
              "### \U0001F4DC The threat", d['narrative'], "",
              "### \U0001F3AF Your task", d['task'], "",
              f"### \U0001F4E5 Source \u2192 \U0001F4E4 Output (Lakehouse `{LH_NAME}`)",
              f"- Read from: `{d['dirty']}`",
              f"- Write the cleaned crystals to: **`{d['output']}`** (exact table name \u2014 the Judge reads this).",
              "",
              "### \U0001F6E0\uFE0F How to build it",
              f"1. Open the **`{d['pipeline']}`** pipeline in the Fabric UI (it was created empty by the installer).",
              f"2. {d['hint']}",
              "3. **Run** the pipeline and wait for it to succeed.",
              "",
              "### \u2705 When done",
              "```python",
              f"judge.check_level({level})",
              "```"]
        display(Markdown("\n".join(md)))

    def check_level(self, level):
        d = LEVELS.get(level)
        if not d:
            print(f"\u2753 unknown level {level}"); return
        log_event("LevelAttempt", level, d['monster'], d['concept'])
        if not _table_exists(d['output']):
            print(f"\u274C Output table `{d['output']}` not found. Build and run `{d['pipeline']}` first,")
            print(f"   making sure its sink writes to the Lakehouse table `{d['output']}`.")
            log_event("LevelFailed", level, d['monster'], d['concept'], validation_result="NO_OUTPUT")
            return
        try:
            passed, rep = diff_tables(d['output'], d['expected'])
        except Exception as e:
            print(f"\u274C Could not compare tables: {e}")
            log_event("LevelFailed", level, d['monster'], d['concept'], validation_result="ERROR")
            return
        print(f"\U0001F50E Inspecting `{d['output']}` vs the secret oracle...")
        print(f"   expected rows: {rep['expected_rows']:,}   your rows: {rep['player_rows']:,}")
        print(f"   missing crystals: {rep['missing']:,}   extra/incorrect: {rep['extra']:,}")
        if passed:
            ev = "BossDefeated" if level == BOSS_LEVEL else "LevelComplete"
            print(f"\n\U0001F3C5 {d['monster']} DEFEATED!  +{d['points']} pts")
            log_event(ev, level, d['monster'], d['concept'], score=d['points'],
                      crystals_saved=rep['player_rows'], rows_processed=rep['player_rows'],
                      validation_result="PASS")
        else:
            print(f"\n\U0001F480 The {d['monster']} still stands. Adjust your pipeline and re-run.")
            log_event("LevelFailed", level, d['monster'], d['concept'],
                      rows_processed=rep['player_rows'], validation_result="FAIL")
        return rep

    def progress(self):
        try:
            rows = query_kql(
                f"{EH_TABLE} | where SessionId == '{SESSION_ID}' "
                f"and EventType in ('LevelComplete','BossDefeated') "
                f"| summarize Score = max(Score), Monster = any(Monster) by Level "
                f"| order by Level asc")
        except Exception as e:
            print(f"\u274C cannot query Eventhouse: {e}"); return
        total = sum(int(r["Score"]) for r in rows)
        defeated = {int(r["Level"]) for r in rows}
        md = ["### \U0001F409 War on Corruption \u2014 this session",
              "| Lvl | Monster | Status | Pts |",
              "|---|---|---|---:|"]
        for n, d in LEVELS.items():
            if n in defeated:
                md.append(f"| {n} | {d['monster']} | \u2705 defeated | {d['points']} |")
            else:
                md.append(f"| {n} | {d['monster']} | \u2014 | 0 |")
        md.append(f"| | | **Total** | **{total}** |")
        md.append(f"\n**Rank:** {rank_for(total)}  ({len(defeated)}/{len(LEVELS)} monsters defeated)")
        if BOSS_LEVEL in defeated:
            md.append("\n> \U0001F3C6 The Corruption King has fallen! Run **Step 4** to mint your badge.")
        display(Markdown("\n".join(md)))
        globals()["FINAL_SCORE"] = int(total)
        globals()["FINAL_RANK"] = rank_for(total)
        return total

judge = Judge()
log_event("QuestStart", 0, "", "", validation_result="OK")
print("\u2696\uFE0F The Judge is ready. Try: judge.help()")

## ▶️ Play

Run `judge.help()`, read a briefing, build the pipeline in the Fabric UI, then check it.

In [ ]:
judge.help()

In [ ]:
# 1. Read the briefing for the level you want to attempt:
judge.briefing(1)

In [ ]:
# 2. After running your pipeline in the Fabric UI, validate it here:
judge.check_level(1)

In [ ]:
# 3. See your overall progress at any time:
judge.progress()

## 🏅 Step 4 — Claim your badge

Defeat the **Corruption King** (level 5) to earn the **Horde Vanquisher** rank and mint a
shareable achievement badge.

In [ ]:
# ============================================================
# Monster Breach - Badge issuance
# HMAC-signed URL for the GitHub Pages badge viewer
# ============================================================
import json, time, hmac, hashlib, base64
from IPython.display import display, Markdown, HTML

_BADGE_SECRET = b"fabric-arcade-badge-v1-7K9mP3xQ"
_BASE_URL     = "https://maenglar78.github.io/fabric-arcade"
_GAME_ID      = "monster-breach"
_SKILLS       = ["Data Pipelines", "Copy Activity", "Dataflow Gen2", "ForEach / If / Switch", "Error Handling"]
_MAX_SCORE    = sum(v["points"] for v in LEVELS.values())

def _b64u(b):
    return base64.urlsafe_b64encode(b).rstrip(b"=").decode("ascii")

def _issue(game_id, player, rank, score):
    payload = {"v": 1, "g": game_id, "p": str(player),
               "r": str(rank), "s": int(score), "t": int(time.time()),
               "k": _SKILLS}
    body = json.dumps(payload, separators=(",", ":"), sort_keys=True).encode()
    sig  = hmac.new(_BADGE_SECRET, body, hashlib.sha256).digest()
    return f"{_BASE_URL}/badge.html?t={_b64u(body)}.{_b64u(sig)}"

score = globals().get("FINAL_SCORE", 0)
rank  = globals().get("FINAL_RANK", "Crystal Recruit")

if score < 100:
    display(Markdown(
        f"### \U0001F6A7 Not yet eligible (score {score}/{_MAX_SCORE})\n\n"
        f"Defeat at least one monster to earn your first badge. Run `judge.check_level(N)` "
        f"on a level you cleared, then `judge.progress()` and this cell again."))
elif PLAYER_NAME.strip() in ("", "Your Name Here"):
    display(Markdown(
        "### \u270D\uFE0F Set your name first\n\n"
        "Edit `PLAYER_NAME` in **Step 0**, re-run `judge.progress()` + this cell."))
else:
    url = _issue(_GAME_ID, PLAYER_NAME, rank, score)
    display(Markdown(
        f"### \U0001F3C5 Badge minted\n\n"
        f"**{PLAYER_NAME}** \u2014 *{rank}* \u00B7 score **{score}/{_MAX_SCORE}**\n\n"
        f"\U0001F517 **[Open your badge]({url})**\n\n"
        f"Click *Download PNG* / *Share on LinkedIn* on the badge page."))
    display(HTML(f'<a href="{url}" target="_blank" '
                 f'style="display:inline-block;padding:10px 20px;border-radius:8px;'
                 f'background:linear-gradient(135deg,#00d4ff,#8338ec);color:white;'
                 f'text-decoration:none;font-weight:600;margin-top:8px;">\U0001F3C5 Open my badge</a>'))